In [ ]:
!pip install transformers datasets sentencepiece torch pandas -q

In [ ]:
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
model_name = "Helsinki-NLP/opus-mt-en-mk"

In [ ]:
print("Вчитување на моделот и токенизаторот...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
print("Вчитување на податоците (Dolly-15k)...")
dataset = load_dataset("databricks/databricks-dolly-15k")
# Земаме само првите 5 примери за тест
small_dataset = dataset["train"].select(range(5))

In [ ]:
def translate_text(text):
    if not text or text.strip() == "":
        return ""

    # Подготовка на текстот за моделот
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    # Генерирање на преводот
    with torch.no_grad():
        translated_tokens = model.generate(**inputs, max_length=512)

    # Декодирање во чист текст
    result = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    return result

In [ ]:
translated_examples = []

In [ ]:
print("Започнува преводот (ова може да потрае неколку секунди)...")
for i, ex in enumerate(small_dataset):
    print(f"Преведување на пример {i+1}...")

    orig_inst = ex["instruction"]
    orig_resp = ex["response"]
    orig_cont = ex.get("context", "")

    mk_inst = translate_text(orig_inst)
    mk_resp = translate_text(orig_resp)
    mk_cont = translate_text(orig_cont) if orig_cont else ""

    # Ги зачувуваме и оригиналните и преведените вредности за табелата
    translated_examples.append({
        "ID": i + 1,
        "orig_instruction": orig_inst,
        "mk_instruction": mk_inst,
        "orig_response": orig_resp,
        "mk_response": mk_resp,
        "orig_context": orig_cont,
        "mk_context": mk_cont
    })

In [ ]:
df = pd.DataFrame(translated_examples)

In [ ]:
pd.set_option("display.max_colwidth", None)

In [ ]:
df = df[["ID", "orig_instruction", "mk_instruction", "orig_response", "mk_response"]]

print("\n" + "="*30 + " ГОТОВО! " + "="*30 + "\n")

In [ ]:
print(df.head())

In [ ]:
#df.to_csv("prevedeni_podatoci.csv", index=False)